In [14]:
import pandas as pd # Data structure for CSV
import re # Regular Expressions
from sklearn.feature_selection import VarianceThreshold
import numpy as np # Numpy need no introduction
from datetime import datetime as dt

the_now = dt.now()
now = the_now.strftime("%Y-%m-%d_%H:%M:%S")

df = pd.read_csv('../dataset/full.csv')
prev_df = pd.read_csv('../dataset/full.csv')
df_columns = df.columns.tolist()
df_cleaned = pd.DataFrame()
prev_df_columns = []

df.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1,STANDING
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1,STANDING
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750,-0.989302,-0.938692,...,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,1,STANDING
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.699205,0.123320,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,1,STANDING


In [15]:
def compare_df():    
    if df_cleaned.empty:
        return
    df_cleaned_columns = df_cleaned.columns.tolist()
    prev_df_columns = prev_df.columns.tolist()

    diff_list_total = list(set(df_columns) - set(df_cleaned_columns))
    diff_list = list(list(set(prev_df_columns) - set(df_cleaned_columns)))

    df_columns_count = len(df_columns)
    df_cleaned_columns_count = len(df_cleaned_columns)
    prev_df_columns_count = len(prev_df_columns)

    total_removed_features = df_columns_count - df_cleaned_columns_count
    this_step_removed_features = prev_df_columns_count - df_cleaned_columns_count

    print("Features excluded (in total):")
    print(diff_list_total)
    print('='*50)
    print("Features excluded (this step):")
    print(diff_list)
    print('='*50)
    print("Removed total: " + str(df_columns_count) + " ---> " + str(df_cleaned_columns_count) + " features")
    print("Total feature removal count: " + str(total_removed_features))
    print('='*50)
    print("Removed this step: " + str(prev_df_columns_count) + " ---> " + str(df_cleaned_columns_count) + " features")
    print("Feature removal this step count: " + str(this_step_removed_features))
    


# Feature extraction/selection

First, we remove features that contain duplicate data to another.


In [16]:
df_cleaned = df.T.drop_duplicates(keep='first').T

df_cleaned.head()


,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.0304,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1,STANDING
1,0.278419,-0.016411,-0.12352,-0.998245,-0.9753,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1,STANDING
2,0.279653,-0.019467,-0.113462,-0.99538,-0.967187,-0.978944,-0.99652,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.98275,-0.989302,-0.938692,...,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,1,STANDING
4,0.276629,-0.01657,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.699205,0.12332,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,1,STANDING


In [17]:
compare_df()
prev_df = df_cleaned

Features excluded (in total):
['tBodyGyroMag-sma()', 'tGravityAccMag-arCoeff()2', 'tGravityAccMag-std()', 'tGravityAccMag-arCoeff()1', 'tGravityAccMag-arCoeff()4', 'fBodyBodyAccJerkMag-sma()', 'tGravityAccMag-min()', 'tGravityAccMag-iqr()', 'tGravityAccMag-entropy()', 'fBodyAccMag-sma()', 'fBodyBodyGyroJerkMag-sma()', 'tGravityAccMag-energy()', 'tGravityAccMag-mean()', 'tBodyAccMag-sma()', 'fBodyBodyGyroMag-sma()', 'tBodyAccJerkMag-sma()', 'tGravityAccMag-max()', 'tGravityAccMag-mad()', 'tGravityAccMag-arCoeff()3', 'tGravityAccMag-sma()', 'tBodyGyroJerkMag-sma()']
Features excluded (this step):
['tBodyGyroMag-sma()', 'tGravityAccMag-arCoeff()2', 'tGravityAccMag-std()', 'tGravityAccMag-arCoeff()1', 'tGravityAccMag-arCoeff()4', 'fBodyBodyAccJerkMag-sma()', 'tGravityAccMag-min()', 'tGravityAccMag-iqr()', 'tGravityAccMag-entropy()', 'fBodyAccMag-sma()', 'fBodyBodyGyroJerkMag-sma()', 'tGravityAccMag-energy()', 'tGravityAccMag-mean()', 'tBodyAccMag-sma()', 'fBodyBodyGyroMag-sma()', 'tBodyAcc

563 ---> 542 features

### Then, we consider what type of feature:
In the dataset, we've got two types of recorded data:
- t-* features: Raw time-series signals recorded at 50hz
- f-* features: Signals transformed using Fast Fourier Transform

There are mathematical functions that have been applied to numerous features, and in some cases the "t" and "p" version of these functions yield no new information. 


## Gemini breakdown because my dumbass dont know shit about physics:

### What you can safely remove:
f*-mean():
This function calculates the plain arithmetic mean of the Fourier amplitudes across all frequency bins. For primary acceleration, this is mathematically tied to the baseline time-domain average (tBodyAcc-mean()) or overall signal power. Dropping these across the board will cause almost zero loss in model accuracy because the time-domain equivalent (t) already captures the signal's overall magnitude

f*.max() + f*.min():
Why f is redundant/invalid: In the time domain, tBodyAcc-max()-X tells you the maximum G-force impulse experienced during a movement step. In the frequency domain, fBodyAcc-max()-X simply tells you the amplitude of the single strongest frequency peak. While peak frequency amplitude is informative, peak frequency intensity is already captured far more comprehensively by fBodyAcc-maxFreqInd (index of peak frequency) or fBodyAcc-energy()

f*.std() + f*.iqr() + f*.energy():
Why f is redundant/invalid: Thanks to Parseval’s Theorem, the total energy (variance) of a signal in the time domain is mathematically equal to the total energy of the signal in the frequency domain. Calculating dispersion statistics like std() or iqr() on the time signal tBodyAcc directly measures movement volatility. Doing the exact same calculation on the FFT bins (fBodyAcc-std()) simply measures how spread out the frequency amplitudes are, which strongly co-varies with time-domain variance.

Source for Parseval's Theorem and its valid application in sensory data feature extraction: 
 10.1109/access.2020.2996576



In [18]:
total_features = df_cleaned.columns.tolist()

# Let's remove all f*-mean():
regex = r"^f.*-mean\(\)"
features_to_remove = [i for i in total_features if re.search(regex, i)]
df_cleaned = df_cleaned.drop(columns=features_to_remove)

# Remove all f*-max():
regex = r"^f.*-max\(\)"
features_to_remove = []
features_to_remove = [i for i in total_features if re.search(regex, i)]
df_cleaned = df_cleaned.drop(columns=features_to_remove)

# Remove all f*-min():
regex = r"^f.*-min\(\)"
features_to_remove = []
features_to_remove = [i for i in total_features if re.search(regex, i)]
df_cleaned = df_cleaned.drop(columns=features_to_remove)

df_cleaned.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.0304,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1,STANDING
1,0.278419,-0.016411,-0.12352,-0.998245,-0.9753,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1,STANDING
2,0.279653,-0.019467,-0.113462,-0.99538,-0.967187,-0.978944,-0.99652,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.98275,-0.989302,-0.938692,...,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,1,STANDING
4,0.276629,-0.01657,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.699205,0.12332,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,1,STANDING


In [19]:
compare_df()
prev_df = df_cleaned

Features excluded (in total):
['fBodyGyro-min()-X', 'fBodyAccJerk-min()-Y', 'tBodyGyroMag-sma()', 'fBodyAccJerk-max()-X', 'fBodyBodyGyroJerkMag-min()', 'fBodyAccJerk-mean()-Y', 'fBodyAccMag-max()', 'tGravityAccMag-arCoeff()2', 'fBodyGyro-min()-Y', 'tGravityAccMag-std()', 'fBodyAccJerk-mean()-X', 'tGravityAccMag-arCoeff()1', 'tGravityAccMag-arCoeff()4', 'fBodyBodyAccJerkMag-sma()', 'fBodyAccMag-mean()', 'tGravityAccMag-min()', 'tGravityAccMag-iqr()', 'fBodyBodyGyroMag-max()', 'tGravityAccMag-entropy()', 'fBodyAccMag-sma()', 'fBodyAcc-mean()-Z', 'fBodyAccJerk-max()-Y', 'fBodyBodyGyroJerkMag-sma()', 'tGravityAccMag-energy()', 'fBodyBodyGyroJerkMag-max()', 'tGravityAccMag-mean()', 'fBodyGyro-max()-X', 'fBodyAcc-min()-Z', 'fBodyGyro-mean()-Y', 'fBodyAcc-max()-Z', 'fBodyBodyAccJerkMag-mean()', 'fBodyAcc-mean()-X', 'fBodyAccJerk-max()-Z', 'tBodyAccMag-sma()', 'fBodyAcc-max()-Y', 'fBodyGyro-min()-Z', 'fBodyAccMag-min()', 'fBodyBodyGyroMag-sma()', 'fBodyBodyAccJerkMag-max()', 'tBodyAccJerkMag-s

542 ---> 503 features

In [20]:
# Remove all f*"-energy" (excluding bandsEnergy)
regex = r"^f.*-energy\(\)"
features_to_remove = []
features_to_remove = [i for i in total_features if re.search(regex, i)]
df_cleaned = df_cleaned.drop(columns=features_to_remove)
df_cleaned.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.0304,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1,STANDING
1,0.278419,-0.016411,-0.12352,-0.998245,-0.9753,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1,STANDING
2,0.279653,-0.019467,-0.113462,-0.99538,-0.967187,-0.978944,-0.99652,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.98275,-0.989302,-0.938692,...,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,1,STANDING
4,0.276629,-0.01657,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.699205,0.12332,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,1,STANDING


In [21]:
compare_df()
prev_df = df_cleaned

Features excluded (in total):
['fBodyGyro-min()-X', 'fBodyAccJerk-min()-Y', 'tBodyGyroMag-sma()', 'fBodyGyro-energy()-Y', 'fBodyAccJerk-max()-X', 'fBodyGyro-energy()-Z', 'fBodyBodyGyroJerkMag-min()', 'fBodyAccJerk-mean()-Y', 'fBodyBodyGyroJerkMag-energy()', 'fBodyAccMag-max()', 'fBodyAcc-energy()-Z', 'tGravityAccMag-arCoeff()2', 'fBodyGyro-min()-Y', 'tGravityAccMag-std()', 'fBodyAccJerk-mean()-X', 'tGravityAccMag-arCoeff()1', 'tGravityAccMag-arCoeff()4', 'fBodyBodyAccJerkMag-energy()', 'fBodyBodyAccJerkMag-sma()', 'fBodyAccMag-mean()', 'tGravityAccMag-min()', 'tGravityAccMag-iqr()', 'fBodyBodyGyroMag-max()', 'tGravityAccMag-entropy()', 'fBodyAccMag-sma()', 'fBodyAcc-mean()-Z', 'fBodyAccJerk-max()-Y', 'fBodyBodyGyroJerkMag-sma()', 'tGravityAccMag-energy()', 'fBodyBodyGyroJerkMag-max()', 'tGravityAccMag-mean()', 'fBodyGyro-max()-X', 'fBodyAcc-min()-Z', 'fBodyAccJerk-energy()-Y', 'fBodyGyro-mean()-Y', 'fBodyAcc-max()-Z', 'fBodyAccMag-energy()', 'fBodyBodyAccJerkMag-mean()', 'fBodyAcc-mean

542 ---> 490 features

# Variance filtering

We are to drop features that have x % sharing of same values.

In [22]:
# We exclude 'subject' and 'Activity':
df_cleaned_no_categorical = df_cleaned.drop(columns=['Activity', 'subject'])

# We'll start easy: 1%.
selector = VarianceThreshold(threshold=0.01)
selector.fit(df_cleaned_no_categorical)

features_to_keep = df_cleaned_no_categorical.columns[selector.get_support()]

df_cleaned = df_cleaned[features_to_keep]

df_cleaned.head()

,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,tBodyAcc-max()-Y,tBodyAcc-max()-Z,tBodyAcc-min()-X,...,fBodyBodyGyroJerkMag-meanFreq(),fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)"
0,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,-0.567378,-0.744413,0.852947,...,-0.074323,-0.298676,-0.710304,-0.112754,0.0304,-0.464761,-0.018446,-0.841247,0.179941,-0.058627
1,-0.998245,-0.9753,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,-0.557851,-0.818409,0.849308,...,0.158075,-0.595051,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317
2,-0.99538,-0.967187,-0.978944,-0.99652,-0.963668,-0.977469,-0.938692,-0.557851,-0.818409,0.843609,...,0.414503,-0.390748,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118
3,-0.996091,-0.983403,-0.990675,-0.997099,-0.98275,-0.989302,-0.938692,-0.576159,-0.829711,0.843609,...,0.404573,-0.11729,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663
4,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,-0.569174,-0.824705,0.849095,...,0.087753,-0.351471,-0.699205,0.12332,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892


In [23]:
compare_df()
prev_df = df_cleaned

Features excluded (in total):
['fBodyAccJerk-min()-Y', 'fBodyAccJerk-bandsEnergy()-57,64', 'fBodyAccJerk-bandsEnergy()-33,48.2', 'fBodyAcc-energy()-Z', 'fBodyAccJerk-mean()-X', 'fBodyAccJerk-bandsEnergy()-25,32.2', 'fBodyGyro-bandsEnergy()-25,48.1', 'fBodyAccJerk-bandsEnergy()-57,64.1', 'tGravityAccMag-energy()', 'tGravityAcc-std()-Y', 'fBodyAcc-bandsEnergy()-33,40.2', 'fBodyGyro-bandsEnergy()-25,32.1', 'fBodyBodyGyroJerkMag-max()', 'fBodyAcc-min()-Z', 'fBodyAccJerk-energy()-Y', 'fBodyAccJerk-max()-Z', 'fBodyAccMag-min()', 'tBodyAccJerkMag-sma()', 'tBodyAcc-mean()-Y', 'fBodyAccJerk-min()-Z', 'fBodyAcc-min()-X', 'fBodyBodyGyroMag-mean()', 'fBodyGyro-bandsEnergy()-41,48.1', 'Activity', 'fBodyBodyGyroMag-energy()', 'fBodyAccJerk-energy()-Z', 'fBodyAccJerk-min()-X', 'fBodyGyro-min()-X', 'tGravityAcc-mad()-X', 'tBodyGyroMag-sma()', 'fBodyAccJerk-bandsEnergy()-57,64.2', 'tGravityAcc-iqr()-Y', 'tGravityAcc-iqr()-X', 'fBodyAccJerk-mean()-Y', 'fBodyBodyGyroJerkMag-energy()', 'fBodyAccMag-max()'

### Working against multicollinearity
But if two features apply to this, which one to drop? We use Target-Aware Correlation Dropping:

In [24]:
threshold = 0.95

corr_matrix = df_cleaned.corr().abs()

# These two segments are Gemini-generated:

# Extract upper triangle of correlation matrix (excluding diagonal)
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find features with correlation higher than threshold
to_drop = [
    column
    for column in upper_tri.columns
    if any(upper_tri[column] > threshold)
]

df_cleaned = df_cleaned.drop(columns = to_drop)


In [25]:
compare_df()
prev_df = df_cleaned



Features excluded (in total):
['fBodyAccMag-mad()', 'fBodyBodyAccJerkMag-iqr()', 'fBodyBodyGyroMag-entropy()', 'tBodyGyro-iqr()-X', 'fBodyAccJerk-bandsEnergy()-49,64', 'fBodyAccJerk-entropy()-Z', 'fBodyAccJerk-mad()-Y', 'fBodyAccJerk-bandsEnergy()-17,24.1', 'fBodyAccJerk-bandsEnergy()-57,64.1', 'fBodyBodyGyroJerkMag-iqr()', 'fBodyAccMag-min()', 'fBodyAcc-min()-X', 'tBodyAccMag-energy()', 'Activity', 'tBodyGyro-mad()-X', 'tBodyAcc-min()-X', 'tGravityAcc-mad()-X', 'tBodyGyroMag-sma()', 'fBodyGyro-mad()-Z', 'fBodyAccMag-max()', 'tBodyGyroMag-std()', 'angle(Z,gravityMean)', 'tBodyGyroJerk-mad()-Y', 'angle(X,gravityMean)', 'tBodyGyroJerk-iqr()-Z', 'tBodyGyroJerk-entropy()-Z', 'tBodyAcc-energy()-X', 'fBodyAccJerk-entropy()-X', 'fBodyAccJerk-mad()-Z', 'fBodyAcc-entropy()-Y', 'fBodyAccJerk-entropy()-Y', 'tBodyAcc-mad()-X', 'fBodyAccJerk-mean()-Z', 'tBodyGyroJerk-min()-Y', 'fBodyGyro-mean()-Z', 'fBodyBodyGyroJerkMag-mean()', 'fBodyGyro-bandsEnergy()-25,48.2', 'tGravityAcc-arCoeff()-X,2', 'tBody

### Bring back 'Activity' and 'subject'
Let's bring the gang back together babey

In [28]:
print(df_cleaned.columns.tolist())

['tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z', 'tBodyAcc-max()-Z', 'tBodyAcc-min()-Z', 'tBodyAcc-energy()-Y', 'tBodyAcc-energy()-Z', 'tBodyAcc-entropy()-X', 'tBodyAcc-entropy()-Y', 'tBodyAcc-entropy()-Z', 'tBodyAcc-arCoeff()-X,1', 'tBodyAcc-arCoeff()-X,2', 'tBodyAcc-arCoeff()-X,3', 'tBodyAcc-arCoeff()-X,4', 'tBodyAcc-arCoeff()-Y,1', 'tBodyAcc-arCoeff()-Y,2', 'tBodyAcc-arCoeff()-Y,3', 'tBodyAcc-arCoeff()-Y,4', 'tBodyAcc-arCoeff()-Z,1', 'tBodyAcc-arCoeff()-Z,2', 'tBodyAcc-arCoeff()-Z,3', 'tBodyAcc-arCoeff()-Z,4', 'tBodyAcc-correlation()-X,Y', 'tBodyAcc-correlation()-X,Z', 'tBodyAcc-correlation()-Y,Z', 'tGravityAcc-mean()-X', 'tGravityAcc-mean()-Y', 'tGravityAcc-mean()-Z', 'tGravityAcc-std()-Z', 'tGravityAcc-sma()', 'tGravityAcc-energy()-Y', 'tGravityAcc-energy()-Z', 'tGravityAcc-entropy()-X', 'tGravityAcc-entropy()-Y', 'tGravityAcc-entropy()-Z', 'tGravityAcc-arCoeff()-X,1', 'tGravityAcc-arCoeff()-Y,1', 'tGravityAcc-arCoeff()-Z,1', 'tGravityAcc-correlation()-X,Y', 'tGravityA

In [27]:
cleaned_df = pd.concat([df_cleaned, df[['Activity', 'subject']]], axis=1)

# cleaned_df.to_csv(rf'../dataset/full.v3_{now}.csv', index=False)